# 03 - Escalamiento lineal

Escala un registro al espectro objetivo por minimos cuadrados logaritmicos en un rango de periodos.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import signalprocessor
print(f"ROOT={ROOT}")
print(f"signalprocessor={Path(signalprocessor.__file__).resolve()}")

In [ ]:
from signalprocessor.io import read_motion_csv, read_spectrum_csv, write_json, write_motion_csv, write_spectrum_csv
from signalprocessor.metrics import motion_metrics
from signalprocessor.plotting import save_spectrum_plot
from signalprocessor.scaling import scale_motion_to_target

OUT = ROOT / "examples" / "ouputs" / "case_03_escalamiento_lineal"
OUT.mkdir(parents=True, exist_ok=True)

motion = read_motion_csv(ROOT / "examples" / "data" / "motion" / "LIMANS.csv", unit="g", name="LIMANS")
target_t, target_sa = read_spectrum_csv(ROOT / "examples" / "data" / "response_spectrum" / "EPU_475.csv")
ouput = scale_motion_to_target(motion, target_t, target_sa, method="log_least_squares", period_range=(0.10, 2.00), factor_bounds=(0.20, 5.00))

write_motion_csv(OUT / "LIMANS_escalado.csv", ouput.motion, unit="g")
write_json(OUT / "LIMANS_escalamiento.json", {"factor": ouput.factor, "method": ouput.method, "metrics": motion_metrics(ouput.motion)})
write_spectrum_csv(OUT / "LIMANS_espectro_escalado.csv", ouput.periods, ouput.scaled_sa_g)
save_spectrum_plot(OUT / "LIMANS_escalamiento.png", ouput.periods, {"registro": ouput.record_sa_g, "escalado": ouput.scaled_sa_g, "objetivo": ouput.target_sa_g}, title=f"LIMANS alpha={ouput.factor:.3f}")

{"factor": ouput.factor, "output": str(OUT)}